# SEVIRI SSL: objectives, training data and results

This notebook follows the [project specification](../handoff_codex_ssl_seviri.md) and compares four learning objectives: **MVM** reconstructs hidden historical observations; **Future** predicts future observations; **Joint** combines reconstruction and forecasting; **Residual** predicts a correction to an advective forecast. The unified figures include all available dataset sizes: **VideoMAE-Small with 4,096, 8,192 and 16,384 clips**, and **VideoMAE-Base with 32,768 clips**. The progress summary and exported inventory record which objectives and analyses were run at each size. Future and Joint are the two objectives trained with 16,384 and 32,768 clips.

**Apparent advection** is the displacement of image structures between observations. **Radiometric evolution** describes changes in their brightness temperature, abbreviated **BT**, which expresses measured radiance as an equivalent blackbody temperature in kelvin. The scientific question is how the training objective changes the predictive information in the learned representations. An observed residual from an advective forecast combines radiometric evolution, errors in estimated motion and observational variability.

The reference forecast **Persistence** repeats the final observed image; **Advection** extrapolates estimated image motion. Both are deterministic forecast algorithms. In probe legends, the name in parentheses identifies the frozen encoder: **Random**, **MVM**, **Future**, **Joint** or **Residual**. Their common forecasting head and fitting data are described in the probe section.

All comparisons use the same **512 validation clips from 17 September 2020**. Training sizes identify numbers of sampled windows and spatial crops. The architecture is stated beside the clip count so the change from Small to Base remains explicit.

## Running and refreshing the study

Select **Python (deep_convection)** and execute the cells in order. The configuration captures the saved progress of every registered run. Comparisons use validation reports from explicit completed epochs. Re-executing the cells refreshes that snapshot.

The default presentation uses figures and explanatory text. **SHOW_TABLES = True** additionally displays the numerical tables; CSV exports retain their exact values in either mode. **USE_WIDGETS = True** enables metric and variability selectors for the channel figures. The notebook reads saved metrics and images. Observation times are UTC; the snapshot timestamp uses Europe/Paris.

In [ ]:
from pathlib import Path
import importlib, json, sys
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Markdown, Image

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/emma_gpm").is_dir())
sys.path.insert(0, str(ROOT / "src"))
SERIES = ROOT / "results/ssl_scaling_20261025"
SERIES_PATHS = [SERIES, ROOT / "results/ssl_extension_20261026", ROOT / "results/ssl_base_32768_20261026"]
OUTPUT = ROOT / "results/notebook_revision_20261028_plots"
OUTPUT.mkdir(exist_ok=True)
USE_WIDGETS = False
SHOW_TABLES = False
SHOW_SPATIAL_COVERAGE = True
if "monitor" in globals():
    monitor.close()
from emma_gpm import ssl_study, ssl_dashboard
study = importlib.reload(ssl_study)
views = importlib.reload(ssl_dashboard)
loaded_studies = [study.load_study(path) for path in SERIES_PATHS if (path / "scaling_plan.json").exists()]
SNAPSHOT = study.merge_studies(*loaded_studies)

def table(frame):
    if not SHOW_TABLES:
        return
    if frame.empty:
        print("This comparison has no saved results in the selected snapshot.")
        return
    formats = {column: "{:.0f}" if pd.api.types.is_integer_dtype(frame[column]) else "{:.2f}"
               for column in frame.select_dtypes(include="number").columns}
    for column in frame.columns:
        if any(term in column.lower() for term in ("epoch", "clips", "updates", "models", "budget")):
            if pd.api.types.is_numeric_dtype(frame[column]):
                formats[column] = "{:.0f}"
        if "(%)" in column or column in ("Mean BT (K)", "BT standard deviation (K)"):
            formats[column] = "{:.1f}"
    display(frame.style.format(formats, na_rep="—").hide(axis="index"))

def figure(fig, filename):
    if fig is not None:
        fig.savefig(OUTPUT / filename, dpi=120, bbox_inches="tight")
        display(fig)
        plt.close(fig)

display(Markdown(f"**Snapshot:** {SNAPSHOT['captured_at'][:19].replace('T', ' ')} Europe/Paris · "
                 f"**Registered series:** {len(loaded_studies)} · **Figures and CSV files:** `{OUTPUT.relative_to(ROOT)}`"))

## Available training results

The registered runs comprise four SSL objectives and five frozen-encoder forecasting probes at each of 4,096 and 8,192 clips, plus Future and Joint at each of 16,384 and 32,768 clips. Saved epochs have committed validation results and checkpoints. Small and Base identify encoder architectures. The completion summary includes every registered run once; detailed progress is exported to `training_status.csv`.

In [ ]:
progress = study.progress_frame(SNAPSHOT)
table(progress)
progress.to_csv(OUTPUT / "training_status.csv", index=False)
completed = sum(run["complete"] for run in SNAPSHOT["runs"].values())
display(Markdown(f"**{completed} of {len(progress)} runs completed.** Saved epochs: {progress['Saved epochs'].sum():,} of {progress['Target epochs'].sum():,}."))

## Forecast accuracy across the available training sets

**RMSE** is the root mean squared difference between predicted and observed brightness temperature, in kelvin. **Global RMSE** pools squared errors over the 512 validation clips, all pixels, seven channels and both lead times before taking the square root. **MAE** averages absolute temperature errors over the same domain. Lower values indicate smaller errors.

The bars compare Future, Joint and Residual, wherever available, with Persistence and Advection. Each training size uses the latest epoch shared by its forecasting objectives: the completed Small runs use 150, 150 and 75 epochs at 4,096, 8,192 and 16,384 clips; Base uses 150 epochs at 32,768 clips. All panels share a vertical scale. **RMSE reduction vs Advection** expresses the reduction relative to the advective RMSE; positive values indicate improvement.

These panels summarize each completed training schedule. The later budget comparison aligns epochs or optimizer updates explicitly. MVM's reconstruction task and its frozen-encoder forecasting probe have dedicated evaluation sections.

In [ ]:
forecast_selection, forecast_coverage = study.select_budget(SNAPSHOT, "pretraining")
forecast_scores = study.metrics_frame(SNAPSHOT, forecast_selection)
forecast_summary = study.forecast_summary(forecast_scores)
table(forecast_coverage)
table(forecast_summary[["Training clips", "Encoder", "Model", "Epoch", "RMSE (K)", "MAE (K)", "RMSE reduction vs Advection (%)"]])
figure(study.overview_figure(forecast_scores), "forecast_overview.png")
forecast_summary.to_csv(OUTPUT / "forecast_summary.csv", index=False)
forecast_scores.to_csv(OUTPUT / "forecast_channel_metrics.csv", index=False)
display(Markdown(study.interpretation(SNAPSHOT, forecast_summary)))

## Observations, sampling units and forecast times

A **clip** is a sampling unit identified by its first timestamp and the origin of a spatial crop. Each clip contains 16 observations separated by five minutes, seven thermal channels and a 224 × 224 pixel crop. Its array shape is `(time, channel, y, x) = (16, 7, 224, 224)`. The elapsed time between its first and last observation is 75 minutes. Every selected clip contains finite observations at all required timestamps, channels and pixels, within the accepted brightness-temperature interval of 100–400 K. The reader uses the coordinate axes to orient each crop with north at the top and west on the left.

The channel order is **WV_062, WV_073, IR_087, IR_097, IR_108, IR_120, IR_134**. These identifiers denote the water-vapour and infrared channels at nominal wavelengths 6.2, 7.3, 8.7, 9.7, 10.8, 12.0 and 13.4 μm. In this notebook, **IR108** is the shorter name for IR_108.

The first ten frames form the historical input, spanning 45 minutes. The timestamp of the tenth frame is the **forecast reference time**. Each forecasting model receives this history and predicts two images, at **+15 and +30 minutes** relative to the reference time. In the stored clip, the historical frames have indices 0–9 and the targets have indices 12 and 15. The intermediate future observations remain available in the archive.

Masked reconstruction uses the same ten historical frames. This common input interval makes the information available to the four encoders comparable. All 16 stored frames contribute to clip validation and to the training-set normalization statistics.

## Nested training sets and temporal separation

The 4,096-clip manifest contains observations from **12 and 13 September 2020**. The 8,192-clip manifest retains those clips and adds observations from **15 September**. The 16,384- and 32,768-clip manifests successively retain all earlier clips and add new combinations of time window and crop origin within those same three dates. The 16,384-clip set contains 5,478 clips from 12 September, 3,619 from 13 September and 7,287 from 15 September. All sizes use the same 512 validation clips from 17 September. The exported dataset metadata record temporal separation and normalization statistics for every size.

Each clip is identified by its first timestamp and crop origin. Overlapping clips share observations and form correlated sampling units. Increasing the number of windows improves sampling of the recorded evolution; additional observation dates would expand the range of atmospheric situations. The 8,192-to-16,384 comparison therefore changes sampling density within the same three days, at the same Small architecture.

The spatial candidate grid has 112-pixel spacing, including the last admissible crop origin at each boundary. Each addition selects half its clips uniformly and half with weights favouring higher historical IR108 variability. Group-selection weights are 1:2:4 for low, medium and high candidate variability. Candidate terciles are calculated within each sampling pool. The variability groups used to evaluate forecasts have fixed reference thresholds, defined below.

Small training runs read lossless files containing complete clips. Base reads four daily NPY arrays containing the original float16 Kelvin values and extracts the requested windows. Both loaders preserve timestamps, spectral order and geographic orientation, verify the stored inputs, and shuffle the order of clips at each epoch.

In [ ]:
rows = []
for case, data in SNAPSHOT["cases"].items():
    protocol, stats = data["protocol"], data["stats"]
    gap = pd.Timestamp(protocol["validation_first_frame"]) - pd.Timestamp(protocol["train_last_frame"])
    minutes = int(gap.total_seconds() / 60)
    rows.append({"Training clips": data["train_samples"], "Validation clips": data["validation_samples"],
        "Training dates (UTC)": ", ".join(protocol["train_days"]),
        "Final training observation (UTC)": pd.Timestamp(protocol["train_last_frame"]).strftime("%d %b %H:%M"),
        "First validation observation (UTC)": pd.Timestamp(protocol["validation_first_frame"]).strftime("%d %b %H:%M"),
        "Temporal separation": f"{minutes // 60} h {minutes % 60:02d} min",
        "Mean BT (K)": stats["global_mean_k"], "BT standard deviation (K)": stats["global_std_k"]})
dataset_metadata = pd.DataFrame(rows)
table(dataset_metadata)
dataset_metadata.to_csv(OUTPUT / "dataset_metadata.csv", index=False)
DATASET_FOR_COVERAGE = "n8192"
if SHOW_SPATIAL_COVERAGE:
    views.dataset_overview(SNAPSHOT["cases"][DATASET_FOR_COVERAGE]["queue"], show_tables=SHOW_TABLES)

### Historical variability and global normalization

The **historical variability score** is the mean absolute IR108 temperature change between successive five-minute observations, averaged over the nine historical frame pairs and all crop pixels. It is expressed in kelvin per observed five-minute difference and responds to transport and radiometric evolution at fixed pixels.

Each dataset size has one global mean and one global standard deviation calculated over every training clip, all 16 frames, seven channels and all crop pixels. Shared observations contribute once for each occurrence in a selected clip. These two constants transform both training and validation observations for that dataset size. The stored precision is used for computation; displayed temperatures and physical errors use rounded values. Reported metrics are converted back to Kelvin units.

## Encoder architectures and four learning objectives

The reference experiments with 4,096, 8,192 and 16,384 clips use **VideoMAE-Small**, with feature dimension 384 and 12 Transformer blocks. Its input is divided into **tubelets**, each containing two frames, a 16 × 16 pixel region and all seven channels. The ten-frame history produces five temporal positions on a 14 × 14 spatial grid, giving 980 tokens.

The forecasting head combines the five temporal feature vectors at each spatial position, applies spatial convolutions and reconstructs two 224 × 224 images with seven channels each. Within each architecture, the objectives start from identical randomly initialized encoder weights, and their forecasting heads share their initial weights. Forecast inputs are restricted to observations at or before the forecast reference time.

### MVM — masked video modeling

**MVM** learns to reconstruct hidden historical brightness temperatures. Tube masking hides 176 of the 196 spatial patch positions throughout the history, covering approximately 90% of the tokens. The encoder receives the visible tokens, and a four-block decoder with feature dimension 192 reconstructs hidden values.

The decoder evaluates 50% of the token positions, using a spatial pattern whose position changes along the temporal sequence. The reconstruction loss is the mean squared error on positions that are both hidden from the encoder and selected for decoding. Training masks vary between presentations of a clip; validation uses a fixed mask per clip.

### Future — causal future-frame prediction

**Future** receives the complete historical input and predicts both future frames directly. The encoder and forecasting head are optimized together using mean squared error across clips, both lead times, all seven channels and all pixels. The targets are the observed images at +15 and +30 minutes.

### Joint — masked reconstruction and future prediction

**Joint** shares one encoder between a masked reconstruction pass and a complete-history forecasting pass. Its objective adds the reconstruction and forecasting losses with equal weights. The reconstruction decoder and forecasting head have separate parameters. The learning curves show their weighted sum as one Joint objective. The saved components also support the historical-reconstruction and forecast evaluations.

### Residual — advection-residual prediction

**Residual** predicts a correction to a fixed advective forecast. The target correction is the observed future image minus that advective forecast. Adding the predicted correction to the advective image produces the final forecast, whose mean squared error drives training. The correction can account for radiometric evolution, spatially varying motion and errors in the estimated transport.
**VideoMAE-Base**, used with 32,768 clips, increases encoder feature dimension from 384 to 768 and uses 12 attention heads per block. It retains 12 Transformer blocks; its four-block reconstruction decoder has feature dimension 384 instead of 192. The forecasting head retains width 192. This experiment changes model capacity, sampling density and the total optimization budget together. At a shared update count, the architecture and training manifest still differ. Separating their contributions requires comparisons that vary one factor at a time.

## Forecast references: Persistence and Advection

**Persistence** copies the final observed image to each future lead time. Its forecast accuracy measures the value of retaining the latest observation.

**Advection** estimates a single horizontal displacement from the last two historical IR108 images using phase correlation, an image-alignment procedure in the Fourier domain. It assigns the same motion vector to every pixel in the crop. Each displacement component is bounded to ±12 pixels per five-minute interval. The algorithm then transports the final image at constant velocity: three displacement steps for +15 minutes and six for +30 minutes. All seven thermal channels use that translation. The algorithm estimates motion independently for each input clip and uses a fixed extrapolation rule.

The **advection support** contains pixels whose translated source positions lie inside the crop. At the other positions, the forecast retains the last observed temperature at the corresponding pixel. This boundary rule supplies a complete forecast and a support mask for evaluation.

**Residual** is the trained experiment associated with this reference: its neural network predicts a temperature correction, which is added to the advected image. The deterministic Advection forecast and the trained Residual forecast appear as separate legend entries because they produce two different predictions. Their difference measures the contribution of the learned correction, including compensation for radiometric evolution, cloud deformation and errors in the uniform-motion approximation.

## Learning trajectories: epochs and optimizer updates

All objectives start from the registered random initialization. Small uses **150 epochs** at 4,096 and 8,192 clips and **75 epochs** at 16,384 clips. Base uses **150 epochs** at 32,768 clips. With effective batch 32, these sizes give **128, 256, 512 and 1,024 optimizer updates per epoch**, respectively. AdamW uses constant learning rate 0.00015 and weight decay 0.05. Computation uses bfloat16; the initialization and data-order seed is 20261024.

The four objectives appear together in each panel. **Colour identifies the objective; line style identifies the encoder architecture and training-set size.** The upper panels show training and the lower panels show validation. The left column uses epochs; the right column uses the recorded optimizer updates. Thus 150 epochs at 8,192 clips and 75 epochs at 16,384 clips both end at 38,400 updates. A matched update count also matches the total number of clip presentations because the effective batch is fixed.

The vertical axis shows each run's **objective loss in K²**, obtained by converting its normalized squared errors with the training-set temperature scale. MVM uses reconstruction error on decoder-selected hidden historical pixels. Future and Residual use future-image error. Joint adds its reconstruction and future-image errors with their registered weights, both equal to one. Joint therefore has one curve for its complete objective. Loss levels are interpreted within each objective; the forecast RMSE panels compare predictive accuracy on a common target. Both axes use logarithmic scales. On the vertical axis, equal distances represent equal multiplicative changes in loss. On the horizontal axis, equal distances represent equal multiplicative changes in epochs or optimizer updates. The first saved epoch and its positive update count define the start of each curve.

Training loss averages batches while parameters evolve during the epoch. Validation uses the weights at the end of the epoch, with fixed historical masks where required. Joint performs two encoder passes per batch. These curves retain every saved epoch, including temporary error peaks.

The original execution accumulated two GPU batches of 16; the revised loader uses one batch of 32. Future at 4,096 and 8,192 clips and MVM at 4,096 completed with the original execution; MVM at 8,192 adopted the revision after epoch 8. Subsequent Small runs use the revised input pipeline. The effective batch is preserved; floating-point reduction order and assignment of random masks to samples can vary with execution grouping.

In [ ]:
figure(study.learning_figure(SNAPSHOT), "learning_trajectories.png")

### Latest error and optimization history

The learning curves retain the full optimization history. `optimization_history.csv` records each run's latest validation RMSE, minimum recorded value and epoch of that minimum. For MVM the quantity is historical-reconstruction error; for Future, Joint and Residual it is forecast error. The overview compares the final registered budget: 150 epochs for 4,096, 8,192 and 32,768 clips, and 75 epochs for 16,384 clips. Minima along a trajectory describe repeated evaluation on the same validation sample. Additional observation periods and seeds would establish reproducibility.

In [ ]:
trajectories = study.trajectory_summary(SNAPSHOT)
table(trajectories)
trajectories.to_csv(OUTPUT / "optimization_history.csv", index=False)
late_windows = []
for (case, name), run in SNAPSHOT["runs"].items():
    if case != "n32768" or name not in ("future", "joint") or len(run["epochs"]) < 60:
        continue
    history = pd.DataFrame(run["epochs"])
    std = SNAPSHOT["cases"][case]["stats"]["global_std_k"]
    for label, window in (("Previous 30 epochs", history.iloc[-60:-30]), ("Last 30 epochs", history.iloc[-30:])):
        late_windows.append({"Training clips": 32768, "Encoder": "Base", "Model": views.LABELS[name],
            "Window": label, "First epoch": int(window.epoch.iloc[0]), "Last epoch": int(window.epoch.iloc[-1]),
            "Mean validation RMSE (K)": window.validation_rmse_k.mean(),
            "Median validation RMSE (K)": window.validation_rmse_k.median(),
            "Training forecast RMSE (K)": window.train_future.mean()**.5 * std})
    a, b = late_windows[-2:]
    display(Markdown(f"**Base / 32,768 · {views.LABELS[name]}:** between epochs {a['First epoch']}–{a['Last epoch']} "
        f"and {b['First epoch']}–{b['Last epoch']}, median validation forecast RMSE changes from "
        f"**{a['Median validation RMSE (K)']:.2f} to {b['Median validation RMSE (K)']:.2f} K**, "
        f"while training forecast RMSE changes from {a['Training forecast RMSE (K)']:.2f} to "
        f"{b['Training forecast RMSE (K)']:.2f} K. These summaries describe late optimization on the same validation day."))
pd.DataFrame(late_windows).to_csv(OUTPUT / "late_training_windows.csv", index=False)


## Effect of dataset size at matched training budgets

The horizontal bars show the **reduction in forecast RMSE when moving from the smaller dataset to the larger dataset**. Positive values favour the larger dataset; negative values indicate higher error with the larger dataset. Each colour identifies a learning objective. Each row compares adjacent dataset sizes, named in its titles. The left column matches epochs and the right column matches optimizer updates. The exact shared budget is included in each panel title. Every plotted value uses two saved validation reports on the same 512 clips.

For Small at 8,192 and 16,384 clips, **38,400 updates** correspond to 150 and 75 epochs. With effective batch 32, both runs process 1,228,800 clip presentations. Their architecture, objectives, learning rate and effective batch agree. The additional windows sample the same three training dates. The right-hand comparison therefore measures the effect of denser sampling at a fixed optimization budget, together with the recorded execution differences.

The Small-to-Base pair changes model capacity as well as clip count. Its largest shared saved update count is 37,888, corresponding to Small epoch 74 and Base epoch 37. The full Base run continues to 153,600 updates. The graph keeps the budget of each comparison explicit. Exact errors, gains and selected epochs are exported to `data_volume_comparison.csv`.

For the frozen probes, expanding from 4,096 to 8,192 clips changes both the encoder pretraining data and the head-fitting data. Their later budget comparison measures that combined procedure; the random-encoder reference helps identify how much head fitting alone benefits from the extra clips.

In [ ]:
volume_comparison = study.size_comparison(SNAPSHOT)
figure(study.size_comparison_figure(volume_comparison), "data_volume_comparison.png")
table(volume_comparison)
volume_comparison.to_csv(OUTPUT / "data_volume_comparison.csv", index=False)

## Errors by channel: all training configurations together

Each row compares the available models and dataset sizes for **one spectral channel**, with +15 minutes on the left and +30 minutes on the right. Bar groups identify architecture and number of training clips; colours identify Future, Joint and Residual, or the frozen encoders when selecting the probe family. The two lead times share their vertical limits within each row. **Persistence** and **Advection** appear as dashed reference lines because they use the same validation observations for every training configuration.

The first row, **All channels**, pools squared errors across all seven channels before taking the square root for RMSE. The remaining rows isolate the individual channels. This makes it possible to compare Small and Base directly within a channel and then assess whether that difference persists in the pooled result. The selected final epochs follow the overview comparison; the preceding budget figure provides the matched-budget comparison.

The default metric is RMSE in kelvin. **Supported RMSE** uses pixels whose advected source positions remain inside the crop. **Gradient MAE** measures error in adjacent-pixel temperature differences, in K/pixel. **Local-minimum MAE** evaluates temperature errors at observed minima in 3 × 3 neighbourhoods, including ties. **Minimum-temperature MAE** compares the coldest predicted and observed temperature in each crop. **Contrast MAE** compares their 95th-to-5th percentile temperature ranges.

**Reference-time tendency MAE** expresses the forecast temperature error per minute of lead time. **Interval tendency MAE** evaluates the first and second consecutive 15-minute changes separately. For these MAE metrics, All channels is the equal-weight mean of the seven channel values; local-minimum counts can differ between channels. RMSE always uses pooled squared errors.

`FAMILY`, `METRIC` and `VARIABILITY` select which saved results are shown. Numerical values and comparison coverage are exported with the figures.

In [ ]:
FAMILY = "pretraining"
METRIC = "rmse_k"
VARIABILITY = "all"  # all, low, medium, high

def show_channels(family, metric, variability):
    selected, coverage = study.select_budget(SNAPSHOT, family)
    scores = study.metrics_frame(SNAPSHOT, selected, variability)
    table(coverage)
    coverage.to_csv(OUTPUT / f"channel_coverage_{family}.csv", index=False)
    if scores.empty:
        print("Channel comparisons are awaiting completed validation results.")
    else:
        figure(study.channel_training_figure(scores, metric), f"channels_{family}_{metric}_{variability}.png")
        study.aggregate_channels(scores).to_csv(OUTPUT / f"channel_comparison_{family}_{variability}.csv", index=False)
        table(study.forecast_summary(scores)[["Training clips", "Model", "Epoch", "RMSE +15 min (K)", "RMSE +30 min (K)"]])

if USE_WIDGETS:
    import ipywidgets as widgets
    family = widgets.Dropdown(options=["pretraining", "frozen_common_head"], value=FAMILY, description="Models:")
    metric = widgets.Dropdown(options=[(label, key) for key, label in views.METRICS.items()], value=METRIC, description="Metric:")
    variability = widgets.Dropdown(options=["all", "low", "medium", "high"], value=VARIABILITY, description="Variability:")
    channel_output = widgets.Output()
    def refresh_channels(*_):
        with views.widget_output(channel_output):
            show_channels(family.value, metric.value, variability.value)
    for control in (family, metric, variability):
        control.observe(refresh_channels, names="value")
    refresh_channels()
    display(widgets.VBox([widgets.HBox([family, metric, variability]), channel_output]))
else:
    show_channels(FAMILY, METRIC, VARIABILITY)

## Does the forecast improvement also cover rapidly changing scenes?

A single global RMSE mixes relatively stable cloud scenes with scenes that change strongly during the input history. The purpose of this diagnostic is to ask **whether a learned model improves on simple transport in both situations**, especially in the rapidly changing scenes relevant to the dynamics question in the project. A model could obtain a useful average score mainly by predicting the stable cases well. Comparing the groups reveals where its improvement occurs and which observations deserve closer inspection or broader sampling.

Each clip's **historical variability score** is the mean absolute IR108 temperature change between successive five-minute observations, averaged over all crop pixels and the nine historical frame pairs. Fixed thresholds of approximately **0.75 and 1.44 K**, estimated from the 4,096-clip training set, divide the validation clips into Low, Medium and High. The same groups are used for every model and training size. The score is calculated from the ten input frames available before forecasting.

The bars now show **percentage reduction in RMSE relative to Advection within each group**. Positive values show added forecast accuracy beyond the estimated uniform transport; zero gives the same RMSE as Advection. A larger positive value in High means that the learned forecast contributes more relative improvement for the rapidly changing input sequences. Absolute errors are retained in the CSV export and in the examples described below. The comparison uses the common final checkpoints from the overview.

For example, if two methods have similar global RMSE but one improves more in High, that method warrants closer examination for rapidly evolving structures. If improvement is concentrated in Low, the aggregate score mainly supports forecasting of stable scenes. Historical changes combine displacement of cloud edges and changes in temperature and shape. The present grouping selects observational variability; identifying convective growth specifically requires a target describing that physical process.

In [ ]:
conditional = study.variability_summary(SNAPSHOT, forecast_selection)
table(conditional[["Variability", "Validation clips"]].drop_duplicates())
figure(study.variability_figure(conditional, "RMSE reduction vs Advection (%)"), "historical_variability.png")
table(conditional[["Training clips", "Variability", "Model", "RMSE (K)", "RMSE reduction vs Advection (%)"]])
conditional.to_csv(OUTPUT / "forecast_by_variability.csv", index=False)
group_sizes = conditional[["Variability", "Validation clips"]].drop_duplicates()
display(Markdown("**Validation groups:** " + "; ".join(f"{row.Variability}: {row['Validation clips']:,} clips" for _, row in group_sizes.iterrows()) + "."))
example = conditional[(conditional["Training clips"] == 32768) & (conditional.Variability == "High")]
if set(["Advection", "Joint"]).issubset(set(example.Model)):
    values = example.set_index("Model")
    display(Markdown(f"For **Joint Base / 32,768**, the High group's RMSE is **{values.loc['Joint', 'RMSE (K)']:.2f} K**, "
        f"compared with **{values.loc['Advection', 'RMSE (K)']:.2f} K** for Advection: "
        f"a **{values.loc['Joint', 'RMSE reduction vs Advection (%)']:.1f}%** reduction on those same validation clips."))

## Temperature errors and errors in cloud structure

Each point combines **two validation errors for one forecast method, one training size, one channel and one lead time**. The horizontal coordinate is image RMSE: the overall mismatch between predicted and observed temperatures, in kelvin. The vertical coordinate is **Gradient MAE**: the mean absolute error in the temperature difference between adjacent pixels, calculated horizontally and vertically, in K/pixel. Both coordinates pool all 512 validation clips. Colours identify methods; circles denote +15 minutes and squares +30 minutes. All panels share their horizontal and vertical limits.

The gradient error describes how well the prediction reproduces local temperature changes across cloud boundaries and internal structures. For example, a forecast that adds the same 2 K offset to every pixel has a temperature error while preserving all neighbouring-pixel differences. A spatially smoothed forecast can reduce some temperature errors while weakening cloud-edge gradients. Errors in the location of a sharp boundary can also increase gradient error. These examples explain why the two coordinates supply complementary information.

A point farther left has lower image RMSE; a point lower down has lower gradient error. Moving toward the lower-left improves both quantities. If one method lies left of another but higher, their ordering depends on the aspect of forecast quality being assessed. The graph places two diagnostics side by side for each prediction; it represents a comparison across methods and lead times.

The exported structural metrics add errors at observed local minima, the coldest temperature in each crop, thermal contrast and the change between consecutive forecast times. These quantities help distinguish temperature accuracy, boundary preservation and cold-extreme preservation. `STRUCTURE_CHANNEL` selects the thermal channel.

In [ ]:
STRUCTURE_CHANNEL = "IR_108"
figure(study.structure_figure(forecast_scores, STRUCTURE_CHANNEL), "radiometry_and_gradients.png")
structural = forecast_scores[forecast_scores.channel == STRUCTURE_CHANNEL].copy()
structural["model"] = structural.model.map(views.LABELS)
columns = ["training_clips", "model", "horizon_minutes", "rmse_k", "gradient_mae_k_per_pixel", "local_minima_mae_k", "minimum_temperature_mae_k", "contrast_p95_p05_mae_k", "interval_tendency_mae_k_per_min"]
structural = structural[columns].rename(columns={"training_clips": "Training clips", "model": "Model", "horizon_minutes": "Lead (min)", **views.METRICS})
table(structural)
structural.to_csv(OUTPUT / "ir108_structural_metrics.csv", index=False)

for size, rows in forecast_scores[forecast_scores.channel == STRUCTURE_CHANNEL].groupby("training_clips"):
    future = rows[(rows.model == "future") & (rows.horizon_minutes == 30)]
    advection = rows[(rows.model == "advection") & (rows.horizon_minutes == 30)]
    if not future.empty and not advection.empty:
        a, b = future.iloc[0], advection.iloc[0]
        display(Markdown(f"At **{size:,} clips** and +30 minutes, Future gives an {STRUCTURE_CHANNEL} image RMSE "
            f"of {a.rmse_k:.2f} K, compared with {b.rmse_k:.2f} K for Advection. "
            f"The error in the coldest temperature within each crop is {a.minimum_temperature_mae_k:.2f} K "
            f"for Future and {b.minimum_temperature_mae_k:.2f} K for Advection. "
            "These two metrics assess average radiometric accuracy and preservation of cold extremes, respectively."))

## Historical reconstruction

This section evaluates the native masked-reconstruction task for **MVM and Joint**. The targets are observed temperatures in the ten historical input frames. Tube masking hides about 90% of spatial patch positions throughout that history. During validation, the decoder evaluates its selected half of the token positions, and the loss uses positions that are both hidden and selected. Masks are fixed per validation clip and shared by the compared models. The bar plot converts this error to **reconstruction RMSE in kelvin** with each dataset's global temperature scale. The x-axis identifies the architecture, training-set size and selected epoch. Bars appear for the trained reconstruction objectives at each size.

The spatial example shows a historical frame from a saved validation clip. **Observation** is the complete measured image. **Masked input** shows the visible observations and marks hidden regions in grey. **Reconstruction** fills those hidden regions with the decoder's predictions while retaining the original observations in visible regions. This combination of predicted and observed pixels is the reason the earlier figure used the term *composite reconstruction*. **Masked-pixel error** shows predicted minus observed temperature only at hidden pixels; positive values indicate a warmer reconstruction and negative values a colder one.

The saved spatial example reconstructs all tokens hidden from the encoder. The quantitative bar plot uses the decoder-selected hidden targets from the validation objective. The figure therefore displays reconstruction across the full mask, while the bars follow the objective recorded during training. All displayed frames precede the forecast reference time. The example selector chooses dataset, channel and historical frame.

In [ ]:
reconstruction, reconstruction_coverage = study.reconstruction_summary(SNAPSHOT)
figure(study.reconstruction_comparison_figure(SNAPSHOT), "historical_reconstruction.png")
reconstruction.to_csv(OUTPUT / "historical_reconstruction.csv", index=False)
study.size_comparison(SNAPSHOT, "reconstruction").to_csv(OUTPUT / "reconstruction_matched_budgets.csv", index=False)

RECONSTRUCTION_DATASET = "n8192"
queue = SNAPSHOT["cases"][RECONSTRUCTION_DATASET]["queue"]
catalog = study.read_json(queue / "analysis/examples/catalog.json")
if catalog:
    MVM_CLIP_INDEX = catalog["indices"][0]
    MVM_CHANNEL = "IR_108"
    HISTORY_FRAME = 0
    if USE_WIDGETS:
        display(views.reconstruction_dashboard(queue))
    else:
        figure(views.reconstruction_figure(queue, MVM_CLIP_INDEX, MVM_CHANNEL, HISTORY_FRAME), "mvm_reconstruction.png")
else:
    print("Spatial reconstruction examples are scheduled with the final analysis after the training series.")

## Forecasting probes: head, data and evaluation

**Current output and target.** Each forecasting probe receives the ten historical frames and outputs **two complete future images**, each with seven channels and a 224 × 224 pixel grid. The output array per clip has shape **(2, 7, 224, 224)**. The corresponding targets are the observed SEVIRI images at **+15 and +30 minutes**, taken from stored clip frames 12 and 15. Training minimizes squared temperature error using the established global normalization; the displayed predictions and metrics are in kelvin.

The common forecasting probes are available for **Small at 4,096 and 8,192 clips**. For each objective, the final pretrained encoder is loaded and its parameters are frozen. A newly initialized spatial forecasting head is then fitted using all historical tokens. The random reference uses the same architecture with its original random encoder weights frozen. Encoder weight checks saved after training confirm that all ten probe encoders remained unchanged.

The fitted head has a **1 × 1 convolution, a 3 × 3 convolution and a transposed convolution**, with GELU activations between the first layers and hidden width 192. It combines the five temporal feature vectors at each spatial patch and reconstructs the two seven-channel future images. For Small, this head has about **1.4 million trainable parameters**. It is a nonlinear spatial decoder. All five encoders at a given size receive exactly the same head architecture, initialization, optimization settings and training budget: **150 epochs**, effective batch 32 and learning rate 0.00015. Its prediction target is the observed image at +15 and +30 minutes.

**Head-fitting observations are the same training clips used for that encoder's pretraining.** The 4,096-clip probes use 12 and 13 September 2020; the 8,192-clip probes also use 15 September. Every probe is evaluated on the same **512 clips from 17 September**, which are temporally separated from those training observations. Within a size, the five heads therefore have identical fitting and validation data. The forecast targets for the heads are computed directly from those observed future frames.

This protocol asks how useful each frozen representation is for fitting a common forecasting decoder, with evaluation on a separate observation day. It supplies a common prediction task for MVM and the predictive encoders, and measures the benefit relative to the frozen random encoder. MVM pretraining reconstructs historical pixels, so future-image prediction already changes its target. Future pretraining and its forecasting probe share their image-prediction target; freezing the encoder and fitting an identical fresh head makes the encoder comparison explicit. Reusing pretraining clips for fitting the head keeps the available fitting observations fixed within each comparison; evidence about prediction on unseen observations comes from the validation day. Future, Joint and Residual pretraining already use future-image targets, while MVM supplies representations learned through historical reconstruction. The resulting scores describe these representations together with the capacity and optimization of the common head.

A stronger transfer experiment would fit every frozen encoder's head on **one common dataset from additional days excluded from all pretraining**, then evaluate on a further untouched period. Keeping that head-fitting dataset and its budget fixed across encoder sizes would isolate the contribution of the pretrained representation more clearly. The current results support the within-protocol comparison above. The repeated use of the present validation day makes an additional final evaluation period useful for confirming the ranking.

**Radiometric ridge outputs and targets.** Each ridge probe outputs **14 scalar values per clip**. Seven targets are the historical mean temperatures of the seven channels. The other seven are historical IR108 variability, historical net temperature change, the final historical frame's mean spatial-gradient magnitude, mean future IR108 temperature change at each of the two lead times, and mean absolute future advection residual at each lead time. These targets are calculated from the observed images. The ridge fits a linear mapping from frozen summary embeddings and uses the same training/validation split. The later figure separates historical observables from future observables.

**Which scientific question comes next?** The current forecasting head measures future-image prediction from fixed representations. To test recovery of convective evolution more directly, a subsequent probe could target cloud-object cooling rate, change in cloud area or a development-stage map, with a common fitting set and an independent evaluation period. RDT-inspired stage labels would measure agreement with the chosen teacher; independent observations would provide a further physical reference. The input can remain SEVIRI while the output and evaluation question change. This is the same general distinction illustrated by [MAE's reconstruction pretraining followed by recognition evaluation](https://openaccess.thecvf.com/content/CVPR2022/papers/He_Masked_Autoencoders_Are_Scalable_Vision_Learners_CVPR_2022_paper.pdf).

In the following plots, the name in parentheses identifies the frozen encoder. **RMSE reduction vs Random** compares a pretrained encoder with the random-encoder probe at the same training size and epoch; positive values indicate improved prediction. The learning curves use the same colour and line-style conventions as the SSL curves. The probe figures cover the two registered Small datasets; detailed coverage is exported as CSV.

In [ ]:
figure(study.learning_figure(SNAPSHOT, "frozen_common_head"), "frozen_probe_learning.png")
probe_selection, probe_coverage = study.select_budget(SNAPSHOT, "frozen_common_head")
probe_scores = study.metrics_frame(SNAPSHOT, probe_selection)
probe_summary = study.forecast_summary(probe_scores)
table(probe_coverage)
probe_coverage.to_csv(OUTPUT / "frozen_probe_coverage.csv", index=False)
table(probe_summary)
if not probe_scores.empty:
    figure(study.overview_figure(probe_scores), "frozen_probe_overview.png")
probe_budgets = study.size_comparison(SNAPSHOT, "frozen_common_head")
figure(study.size_comparison_figure(probe_budgets), "frozen_probe_matched_budgets.png")
probe_budgets.to_csv(OUTPUT / "frozen_probe_matched_budgets.csv", index=False)
probe_summary.to_csv(OUTPUT / "frozen_probe_summary.csv", index=False)

probe_reference = study.probe_reference_summary(probe_summary)
figure(study.probe_reference_figure(probe_reference), "frozen_probe_vs_random.png")
table(probe_reference)
probe_reference.to_csv(OUTPUT / "frozen_probe_vs_random.csv", index=False)
for size, rows in probe_reference.groupby("Training clips"):
    learned = rows[rows.Model != "Probe (Random)"].sort_values("RMSE (K)")
    if learned.empty:
        continue
    better = learned[learned["RMSE reduction vs Random (K)"] > 0]
    higher = learned[learned["RMSE reduction vs Random (K)"] <= 0]
    statements = []
    if not better.empty:
        values = "; ".join(f'{row.Model}: {row["RMSE reduction vs Random (K)"]:.2f} K lower RMSE' for _, row in better.iterrows())
        statements.append(f"At {size:,} clips, the pretrained encoders improving on the Random reference are {values}.")
    if not higher.empty:
        values = "; ".join(f'{row.Model}: {-row["RMSE reduction vs Random (K)"]:.2f} K higher RMSE' for _, row in higher.iterrows())
        statements.append(f"The other pretrained encoders give {values}.")
    display(Markdown(" ".join(statements)))

## Saved forecast examples

Every completed training run has a saved validation figure for the first clip in the shared validation manifest. The figures below compare that same observation for Future, Joint and Residual at the selected training size. The model is named immediately above its figure; **neural** inside the saved figure denotes that model's forecast. Each figure includes the last input, the future observation, Persistence and Advection at both lead times. Brightness temperatures share the 180–330 K colour scale, with north at the top and west on the left.

The final analysis also saves a four-clip example catalog spanning the validation variability range, with reusable prediction arrays and error maps. The catalog is used by the interactive forecast and reconstruction views once it is available.

In [ ]:
EXAMPLE_DATASET = "n8192"
for model in ("future", "joint", "residual"):
    run = SNAPSHOT["runs"].get((EXAMPLE_DATASET, model))
    if run is None:
        continue
    path = run["directory"] / "forecast_latest.png"
    if run["complete"] and path.exists():
        display(Markdown(f"**{views.LABELS[model]} · {SNAPSHOT['cases'][EXAMPLE_DATASET]['train_samples']:,} training clips · epoch {run['completed_epochs']}**"))
        display(Image(filename=str(path), width=1100))
    else:
        print(f"{views.LABELS[model]}: the final-epoch forecast image is awaiting completion.")

## Temporal interventions and representation analysis

The saved representation analyses cover the 4,096- and 8,192-clip Small experiments. Their availability is recorded alongside the later forecasting runs in the exported diagnostic inventory.

### Response to changes in the input history

The temporal comparisons change the encoder input while keeping model weights and forecast targets fixed. **Chronological** uses the ten historical frames in their observed order. **Shuffled history** permutes the first nine frames reproducibly and retains the final observed frame in its original position. **Last frame repeated** uses ten copies of that final frame.

Persistence and Advection are calculated from the original history for every comparison. Residual also retains its original advective component. The resulting error changes measure how the neural prediction responds to historical content and ordering under a fixed transport reference. Both temporal information and the change in input distribution contribute to this response.

The embedding comparison adds **Advection-only history**, a synthetic sequence formed by repeatedly translating the first historical image with the displacement estimated from the last two real historical IR108 images. Positions outside the crop retain the first image's value at the corresponding pixel. The saved diagnostics include the fraction of pixels supported by the translation.

### Representation summaries and radiometric probes

The representation analysis first averages the encoder features over the spatial grid, producing one 384-component vector at each of the five temporal positions. It concatenates the temporal mean of these vectors with their signed linear temporal trend, giving a 768-component embedding. The trend retains the direction of feature changes across the history. Ridge probes and embedding-distance comparisons use this summary; the common forecasting probes use the full spatial feature grid.

The **Embedding sensitivity** plot shows mean cosine distance between the original and altered embeddings over the 512 validation clips. Larger distances indicate a larger change in feature-vector direction under the selected input intervention. Forecasting and radiometric probes assess the predictive information available in those features.

The ridge analysis uses the same regularization strength of 10 for all five encoders. Embedding components are centred and scaled using training-set statistics, and regression coefficients are fitted on the training clips before evaluation on validation.

There are 14 radiometric targets. Seven are historical brightness-temperature means, one per channel, averaged over the ten input frames and all crop pixels. The remaining seven describe IR108 dynamics and structure: historical variability; the mean temperature change across the 45-minute history, expressed as a rate in K per 15 minutes; the final input's mean absolute spatial gradient, in K/pixel; the mean temperature change from the final input to each of the two future images; and the mean absolute difference between the observed and advected image at each future lead time. In the ridge plots, **residual amplitude** denotes this last quantity in kelvin.

The dot plots focus on these seven dynamic and structural targets and report validation **R²**, the coefficient of determination. A value of 1 indicates exact predictions, 0 matches the squared error obtained by predicting the validation mean, and negative values indicate greater error than that reference. The saved results also include MAE and the error obtained by predicting the training-target mean.

### Availability in the current series

The exported diagnostic inventory identifies the analysis products already saved for each dataset size. The supervisor runs this analysis after the scheduled training runs. Existing current-series products are displayed when available; the report retains their model and intervention labels.

The temporal plots compare global RMSE under each intervention with chronological history. Exact changes for every model and training size are exported in `temporal_rmse_changes.csv`, using the same validation observations. The radiometric interpretation below focuses on predicting the signed mean temperature change of the whole crop and the amplitude of the advection residual; these are distinct forecast targets.

In [ ]:
availability = study.diagnostic_availability(SNAPSHOT)
table(availability)
availability.to_csv(OUTPUT / "diagnostic_availability.csv", index=False)
for case, data in SNAPSHOT["cases"].items():
    if (data["queue"] / "analysis/temporal_ablations.csv").exists():
        display(Markdown(f"**Representation diagnostics · {data['train_samples']:,} training clips**"))
        views.representation_diagnostics(data["queue"], show_tables=SHOW_TABLES)

temporal_rows = []
for case, data in SNAPSHOT["cases"].items():
    analysis = data["queue"] / "analysis"
    if study.read_json(analysis / "progress.json").get("status") != "completed":
        continue
    temporal = pd.read_csv(analysis / "temporal_ablations.csv")
    pivot = temporal.pivot(index="run", columns="ablation", values="rmse_k")
    for model, values in pivot.iterrows():
        temporal_rows.append({"Training clips": data["train_samples"], "Model": views.LABELS[model],
            "Chronological RMSE (K)": values["chronological"],
            "Shuffled-history increase (K)": values["shuffle"] - values["chronological"],
            "Repeated-last-frame increase (K)": values["last_frame"] - values["chronological"]})
    ridge = pd.read_csv(analysis / "radiometric_probes.csv")
    for minutes in (15, 30):
        target = f"future_ir108_mean_change_{minutes}min_k"
        rows = ridge[ridge.descriptor == target]
        best = rows.loc[rows.r2.idxmax()]
        if (rows.r2 < 0).all():
            display(Markdown(f"With **{data['train_samples']:,} clips**, all five ridge probes have negative validation R² "
                f"for the crop's signed mean IR108 change at +{minutes} minutes. "
                f"The highest R² is {best.r2:.2f}, obtained by {views.LABELS[best.run]}. "
                "Their squared errors exceed the reference obtained from the validation-target mean."))
    residual = ridge[ridge.descriptor.str.startswith("future_ir108_advection_residual_mae_")]
    winners = [group.loc[group.r2.idxmax()] for _, group in residual.groupby("descriptor")]
    if len(winners) == 2 and all(row.run == "probe_random" for row in winners):
        values = ", ".join(f"+{15 if '15min' in row.descriptor else 30} minutes: R² {row.r2:.2f}" for row in winners)
        display(Markdown(f"At the same training size, **Probe (Random)** has the highest ridge R² "
            f"for future residual amplitude at both lead times ({values}). "
            "The common forecasting head and these ridge probes therefore reveal different aspects of the representations: "
            "prediction of full spatial images and linear recovery of crop-level quantities."))
temporal_comparison = pd.DataFrame(temporal_rows)
table(temporal_comparison)
temporal_comparison.to_csv(OUTPUT / "temporal_rmse_changes.csv", index=False)

## Computational measurements

The separate [performance notebook](profile_seviri_ssl_performance.ipynb) records data preparation, GPU execution, validation, checkpoint timing, memory and CUDA activity. Its complete-epoch measurements compare the original and revised input pipelines. The exported performance comparison records those measurements, whose timings exclude initialization and profiler-recorded epochs. The current complete-epoch duration includes checkpoint persistence; the previous column sums the two components recorded in the historical logs.

The scientific comparison above retains each run's registered learning rate, effective batch and training budget. Execution revisions and their checkpoint lineage are recorded with the series. Joint's extra reconstruction pass contributes additional computational work per optimizer update.

In [ ]:
performance_path = SERIES / "execution_revisions/clip_files_v1/comparison.json"
if performance_path.exists():
    performance = study.read_json(performance_path)
    measured = pd.DataFrame(performance["timing_rows"])
    formats = {column: "{:.0f}" for column in measured.columns if column.endswith("(s)")}
    measured.to_csv(OUTPUT / "performance_comparison.csv", index=False)
    if SHOW_TABLES:
        display(measured.style.format(formats).hide(axis="index"))

## Reproducibility and saved products

The unified snapshot reads `results/ssl_scaling_20261025/`, `results/ssl_extension_20261026/` and `results/ssl_base_32768_20261026/`. Each dataset directory contains its frozen manifests, model runs and available analyses. Before loading, the reader verifies the manifests and normalization statistics against the registered case plans, checks the common validation manifest and confirms strict nesting of training clips. Validation results come from the explicit completed epochs in `validation_epochs/epoch_*.json`. Global RMSE pools squared errors across channels and lead times before taking the square root.

The current figures, numerical CSV exports and executed notebook are stored in `results/notebook_revision_20261028_plots/`. Earlier review outputs remain in their original result directories. The source notebook contains clean code and explanations; `study_seviri_ssl_scaling.ipynb` links to this unified study. The saved files for each run preserve its original training configuration and checkpoint history.

Interpretation currently rests on one initialization seed, two or three training dates and one shared validation date. Additional dates, seeds, a common independent dataset for fitting probes, a separate final evaluation period and physical validation of the motion approximation would strengthen conclusions about reproducibility and atmospheric dynamics. Further planned controls include alternative masking schemes and a separately trained single-frame model.

The [project specification](../handoff_codex_ssl_seviri.md) defines the scientific questions. [SSL_SEVIRI.md](../SSL_SEVIRI.md) records preparation, training, analysis and remote-monitoring commands.